In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
order_items = pd.read_csv("/content/drive/MyDrive/Sales/order_items.csv")
orders = pd.read_csv("/content/drive/MyDrive/Sales/orders.csv")
customers = pd.read_csv("/content/drive/MyDrive/Sales/customers.csv")
products = pd.read_csv("/content/drive/MyDrive/Sales/products.csv")

In [ ]:
sales_master = orders.merge(order_items,how = 'outer')

In [ ]:
sales_master = sales_master.merge(customers,how='outer')

In [ ]:
sales_master = sales_master.merge(products,how='outer')

In [ ]:
sales_master.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 113425 entries, 0 to 113424
Data columns (total 26 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   order_id                       113425 non-null  object 
 1   customer_id                    113425 non-null  object 
 2   order_status                   113425 non-null  object 
 3   order_purchase_timestamp       113425 non-null  object 
 4   order_approved_at              113264 non-null  object 
 5   order_delivered_carrier_date   111457 non-null  object 
 6   order_delivered_customer_date  110196 non-null  object 
 7   order_estimated_delivery_date  113425 non-null  object 
 8   order_item_id                  112650 non-null  float64
 9   product_id                     112650 non-null  object 
 10  seller_id                      112650 non-null  object 
 11  shipping_limit_date            112650 non-null  object 
 12  price                         

In [ ]:
sales_master.dtypes

,0
order_id,object
customer_id,object
order_status,object
order_purchase_timestamp,object
order_approved_at,object
order_delivered_carrier_date,object
order_delivered_customer_date,object
order_estimated_delivery_date,object
order_item_id,float64
product_id,object


In [ ]:
sales_master.shape

(113425, 26)

In [ ]:
date_col = ['order_approved_at','order_purchase_timestamp','order_delivered_carrier_date','order_delivered_customer_date',
            'order_estimated_delivery_date', 'shipping_limit_date']
sales_master[date_col] = sales_master[date_col].apply(
    pd.to_datetime,
    errors = 'coerce'
)

In [ ]:
sales_master['revenue'] = sales_master['price'] + sales_master['freight_value']

In [ ]:
print(sales_master)

                                order_id                       customer_id  \
0       f30149f4a8882a08895b6a242aa0d612  86c180c33f454b35e1596a99da3dddc4   
1       f5eda0ded77c1293b04c953138c8331d  68f2b37558e27791155db34bcded5ac0   
2       0bf736fd0fd5169d60de3699fcbcf986  6cd217b674e22cf568f6a2cf6060fd07   
3       3aba44d8e554ab4bb8c09f6f78032ca8  82b838f513e00463174cc7cae7e76c1f   
4       6f0dfb5b5398b271cc6bbd9ee263530e  8517e7c86998bf39a540087da6f115d9   
...                                  ...                               ...   
113420  87ae60ef8b08ae0e5f903cacb53a6904  fea4d57ed3a45455f89c25ef3dae8ee8   
113421  d69fcef5a5fe3a3db60ea65c6ee499cc  ff0340330283c0d05ea7281e67fa2e76   
113422  f86d7bc39aab05299691322044b63bb2  ff31bee1ba4bba00cc14b1b91c8d28f3   
113423  74f833bf7c70ce8c3f820f725c213e1c  fffc22669ca576ae3f654ea64c8f36be   
113424  2e935fa1d39497aa0ec3f1107fbfb5b8  ffffe8b65bbe3087b653a978c870db99   

       order_status order_purchase_timestamp   order_approved_a

In [ ]:
sales_master['delivery_days'] = (sales_master['order_delivered_customer_date'] - sales_master['order_purchase_timestamp']).dt.days

In [ ]:
sales_master['is_late'] = np.where(sales_master['order_delivered_customer_date'] > sales_master['order_estimated_delivery_date'],1,0)

In [ ]:
sales_master[['is_late','order_delivered_customer_date','order_estimated_delivery_date']].tail(10)

,is_late,order_delivered_customer_date,order_estimated_delivery_date
113415,0,NaT,2017-03-13
113416,0,NaT,2017-09-01
113417,0,NaT,2018-03-15
113418,0,NaT,2018-01-22
113419,0,NaT,2017-11-08
113420,0,NaT,2018-09-26
113421,0,NaT,2018-01-08
113422,0,NaT,2018-09-25
113423,0,NaT,2017-08-01
113424,0,NaT,2017-10-24


In [ ]:
sales_master['order_month'] = sales_master['order_purchase_timestamp'].dt.month

In [ ]:
print(sales_master)

In [ ]:
sales_master['order_status'].unique()

array(['delivered', 'processing', 'shipped', 'invoiced', 'canceled',
       'unavailable', 'approved', 'created'], dtype=object)

In [ ]:
sales_master[['order_purchase_timestamp','order_month']]

,order_purchase_timestamp,order_month
0,2018-05-20 18:45:21,5
1,2017-12-12 19:20:28,12
2,2017-12-21 16:21:47,12
3,2018-08-10 13:24:35,8
4,2018-08-01 22:00:33,8
...,...,...
113420,2018-09-11 16:45:54,9
113421,2017-12-02 10:11:44,12
113422,2018-08-13 18:43:06,8
113423,2017-06-30 11:21:11,6


In [ ]:
sales_master = sales_master[
    sales_master['order_status'] != 'canceled'
].reset_index(drop = True)

In [ ]:
sales_master['order_status'].unique()

array(['delivered', 'processing', 'shipped', 'invoiced', 'unavailable',
       'approved', 'created'], dtype=object)

In [ ]:
sales_master

In [ ]:
sales_master = sales_master.drop_duplicates(subset = 'order_id')

In [ ]:
print(sales_master.isnull().sum())

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                  19
order_delivered_carrier_date     1233
order_delivered_customer_date    2346
order_estimated_delivery_date       0
order_item_id                     611
product_id                        611
seller_id                         611
shipping_limit_date               611
price                             611
freight_value                     611
customer_unique_id                  0
customer_zip_code_prefix            0
customer_city                       0
customer_state                      0
product_category_name            2017
product_name_lenght              2017
product_description_lenght       2017
product_photos_qty               2017
product_weight_g                  627
product_length_cm                 627
product_height_cm                 627
product_width_cm                  627
revenue     

In [ ]:
null_index = sales_master[
sales_master['product_length_cm'].isnull()
].index
print(null_index)

Index([  4051,  42802,  42803,  42804,  42805,  42806,  42807,  42808,  42809,
        42811,
       ...
       112709, 112710, 112711, 112712, 112713, 112714, 112715, 112716, 112717,
       112718],
      dtype='int64', length=627)


In [ ]:
for i in null_index:
  print(sales_master.loc[i-5:i+5,'product_length_cm'])

Streaming output truncated to the last 5000 lines.
112297   NaN
112298   NaN
112299   NaN
112300   NaN
112301   NaN
112302   NaN
112303   NaN
112304   NaN
112305   NaN
112306   NaN
Name: product_length_cm, dtype: float64
112297   NaN
112298   NaN
112299   NaN
112300   NaN
112301   NaN
112302   NaN
112303   NaN
112304   NaN
112305   NaN
112306   NaN
112307   NaN
Name: product_length_cm, dtype: float64
112298   NaN
112299   NaN
112300   NaN
112301   NaN
112302   NaN
112303   NaN
112304   NaN
112305   NaN
112306   NaN
112307   NaN
112308   NaN
Name: product_length_cm, dtype: float64
112299   NaN
112300   NaN
112301   NaN
112302   NaN
112303   NaN
112304   NaN
112305   NaN
112306   NaN
112307   NaN
112308   NaN
112309   NaN
Name: product_length_cm, dtype: float64
112300   NaN
112301   NaN
112302   NaN
112303   NaN
112304   NaN
112305   NaN
112306   NaN
112307   NaN
112308   NaN
112309   NaN
112310   NaN
Name: product_length_cm, dtype: float64
112301   NaN
112302   NaN
112303   NaN
112304  

In [ ]:
sales_master.dropna(subset = ['product_length_cm'],inplace = True)

In [ ]:
sales_master['product_category_name']

In [ ]:
sales_master = sales_master[
    sales_master['order_status'] != 'unavailable'
].reset_index(drop = True)

In [ ]:
sales_master['order_approved_at'] = (sales_master['order_approved_at'].fillna(sales_master['order_purchase_timestamp']))

In [ ]:
print(sales_master)

In [ ]:
sales_master.dropna(subset= ['product_category_name'],inplace = True)

In [ ]:
sales_master.dropna(subset= ['order_delivered_customer_date'],inplace = True)

In [ ]:
sales_master.dropna(subset = ['order_delivered_carrier_date'],inplace = True)

In [ ]:
sales_master.reset_index(drop = True, inplace = True)

In [ ]:
text_cols = sales_master.select_dtypes(include = 'object').columns
sales_master[text_cols] = sales_master[text_cols].apply(lambda x : x.str.strip())

In [ ]:
print(sales_master)

In [ ]:
sales_master.to_excel(
    '/content/drive/MyDrive/sales_master.xlsx',
    index=False
)